# Chest X-ray Expert — Training & Evaluation

**What this notebook does:**
1. Pulls the Doctor-Assistant codebase from GitHub
2. Downloads NIH ChestX-ray14 via the Kaggle API
3. Trains `build_chest_xray_expert` (DenseNet-121 backbone, 14 multi-label pathologies)
4. Evaluates with per-class AUC + clinical metrics
5. Generates a sample structured radiology report

**Runtime:** Use **L4 GPU** (24 GB VRAM) for batch_size=64 at 320×320. A100 also works.  
**Checkpoints** are written to Google Drive after every epoch — safe against session timeouts.

## 0 · GPU & Drive

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — change runtime to L4/A100"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name}  VRAM: {gpu.total_memory / 1e9:.1f} GB")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT   = '/content/drive/MyDrive/doctor_assistant'
CKPT_DIR     = f'{DRIVE_ROOT}/checkpoints/chest_xray'
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/chest_xray14'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATASET_ROOT, exist_ok=True)
print('Drive mounted. Checkpoint dir:', CKPT_DIR)

## 1 · Pull codebase from GitHub

In [ ]:
REPO_URL  = 'https://github.com/Hamza09Hamza/doctor_assistant.git'
REPO_DIR  = '/content/doctor_assistant'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo ready.')

## 2 · Install dependencies

In [ ]:
%%capture
!pip install -q \
    'monai>=1.3' \
    'nibabel>=5.2' \
    'pydicom>=2.4' \
    'SimpleITK>=2.3' \
    'timm>=0.9' \
    'scikit-image>=0.22' \
    'scikit-learn>=1.4' \
    'pydantic>=2.6'
print('Dependencies installed.')

## 3 · Download NIH ChestX-ray14

**One-time setup:** upload your `kaggle.json` API token (Account → API → Create Token).  
The dataset (~42 GB) is downloaded to Drive so subsequent sessions skip this step.

In [ ]:
IMAGES_DIR = os.path.join(DATASET_ROOT, 'images')
CSV_PATH   = os.path.join(DATASET_ROOT, 'Data_Entry_2017.csv')

if os.path.isfile(CSV_PATH) and os.path.isdir(IMAGES_DIR):
    n_images = len(os.listdir(IMAGES_DIR))
    print(f'Dataset already on Drive: {n_images:,} images found. Skipping download.')
else:
    print('Dataset not found on Drive. Starting download via Kaggle API...')
    from google.colab import files
    print('Upload your kaggle.json:')
    uploaded = files.upload()
    !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    # Download to a local temp dir, then move to Drive
    LOCAL_TMP = '/content/nih_tmp'
    !kaggle datasets download -d nih-chest-xrays/data -p {LOCAL_TMP} --unzip

    # The Kaggle package unzips into images_001/ ... images_012/ subdirs; flatten them.
    print('Flattening image subdirectories...')
    import glob, shutil
    os.makedirs(IMAGES_DIR, exist_ok=True)
    for src in glob.glob(f'{LOCAL_TMP}/images_*/*.png'):
        shutil.move(src, IMAGES_DIR)
    # Copy metadata files
    for f in ['Data_Entry_2017.csv', 'train_val_list.txt', 'test_list.txt']:
        src = os.path.join(LOCAL_TMP, f)
        if os.path.isfile(src):
            shutil.copy(src, DATASET_ROOT)
    shutil.rmtree(LOCAL_TMP, ignore_errors=True)
    print(f'Done. {len(os.listdir(IMAGES_DIR)):,} images in Drive.')

## 4 · Config — tune these before training

In [ ]:
# ─── Training hypers ───────────────────────────────────────────────────────
BACKBONE    = 'timm:densenet121'   # CheXNet standard
IMAGE_SIZE  = 320                  # 320×320; go 224 if VRAM is tight
BATCH_SIZE  = 64                   # L4 24GB handles 64 at 320px with AMP
EPOCHS      = 20
LR          = 1e-4                 # AdamW; cosine decay to LR_MIN
LR_MIN      = 1e-6
WEIGHT_DECAY= 1e-5
FREEZE_EPOCHS = 2                  # freeze backbone for first N epochs (head warm-up)
NUM_WORKERS = 4
VAL_FRACTION= 0.1                  # 10% of train_val_list goes to val

# ─── Paths (already set above) ─────────────────────────────────────────────
print(f'Backbone: {BACKBONE}  |  {IMAGE_SIZE}px  |  bs={BATCH_SIZE}  |  {EPOCHS} epochs')

## 5 · Build the expert + data loaders

In [ ]:
from experts.chest_xray import build_chest_xray_expert
from data.chest_xray14 import CHESTXRAY14_LABELS, dataset_stats, load_chest_xray14
from data.dataset import ScanDataset
from core.enums import BodyPart, Modality
from preprocessing.transforms import PreprocessConfig, build_preprocess
from torch.utils.data import DataLoader

# ── Expert model ──
expert = build_chest_xray_expert(
    backbone=BACKBONE,
    image_size=IMAGE_SIZE,
    pretrained=True,
)
n_params = sum(p.numel() for p in expert.parameters()) / 1e6
print(f'Expert: {expert.name}  |  {n_params:.1f}M params')

# ── Preprocessing pipelines (train augments, val does not) ──
train_cfg = PreprocessConfig(
    spatial_size=(IMAGE_SIZE, IMAGE_SIZE), in_channels=3, intensity='scale', augment=True
)
val_cfg = PreprocessConfig(
    spatial_size=(IMAGE_SIZE, IMAGE_SIZE), in_channels=3, intensity='scale', augment=False
)
train_pre = build_preprocess(train_cfg, train=True)
val_pre   = build_preprocess(val_cfg,   train=False)

# ── Samples ──
print('Loading sample lists from CSV...')
train_samples = load_chest_xray14(DATASET_ROOT, split='train', val_fraction=VAL_FRACTION)
val_samples   = load_chest_xray14(DATASET_ROOT, split='val',   val_fraction=VAL_FRACTION)
print(f'Train: {len(train_samples):,}  Val: {len(val_samples):,}')

stats = dataset_stats(train_samples)
print('\nTrain label counts:')
for label, count in sorted(stats.per_label.items(), key=lambda x: -x[1]):
    print(f'  {label:<22} {count:>7,}  ({count/stats.total*100:.1f}%)')
print(f'  No Finding             {stats.no_finding:>7,}')

# ── Datasets + loaders ──
train_ds = ScanDataset(train_samples, train_pre, modality=Modality.XRAY, body_part=BodyPart.CHEST)
val_ds   = ScanDataset(val_samples,   val_pre,   modality=Modality.XRAY, body_part=BodyPart.CHEST)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'\nBatches per epoch — train: {len(train_loader):,}  val: {len(val_loader):,}')

## 6 · Trainer setup

In [ ]:
from training.trainer import Trainer, TrainConfig
from training.losses import MultiTaskLoss
from evaluation.multilabel_metrics import MultilabelEvaluator
import torch

evaluator = MultilabelEvaluator(class_names=list(CHESTXRAY14_LABELS))

cfg = TrainConfig(
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    mixed_precision=True,
    grad_clip=1.0,
    monitor='auc',             # macro AUC selects best checkpoint
    checkpoint_dir=CKPT_DIR,
    resume=True,               # auto-resume from last.pt on session restart
)

optimizer = torch.optim.AdamW(expert.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Cosine annealing: smoothly decays LR from LR to LR_MIN over all epochs.
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=LR_MIN
)

trainer = Trainer(
    model=expert,
    evaluator=evaluator,
    config=cfg,
    loss_fn=MultiTaskLoss(multilabel=True),
    optimizer=optimizer,
    scheduler=scheduler,
)
print(f'Trainer ready. Device: {trainer.device}  AMP: {trainer.use_amp}')

## 6b · Backbone freeze warm-up (optional but recommended)

Freeze the DenseNet-121 trunk for the first `FREEZE_EPOCHS` epochs so the randomly-initialised classification head converges before the pretrained weights are disturbed. Then unfreeze for full fine-tuning.

In [ ]:
def freeze_backbone(model, frozen: bool) -> None:
    for p in model.backbone.parameters():
        p.requires_grad = not frozen
    status = 'frozen' if frozen else 'unfrozen'
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
    print(f'Backbone {status}. Trainable params: {trainable:.1f}M')

freeze_backbone(expert, frozen=True)

## 7 · Train

In [ ]:
history = []

for epoch in range(EPOCHS):
    # Unfreeze backbone after warm-up
    if epoch == FREEZE_EPOCHS:
        print(f'\n--- Epoch {epoch}: unfreezing backbone for full fine-tuning ---')
        freeze_backbone(expert, frozen=False)

    # Run one epoch via trainer internals
    train_loss = trainer._train_one_epoch(train_loader)
    metrics    = trainer._validate(val_loader)
    score      = trainer._monitored_score(metrics)

    trainer._save('last.pt', epoch)
    if score > trainer.best_metric:
        trainer.best_metric = score
        trainer._save('best.pt', epoch)
        print(f'  ★ new best AUC {score:.4f} — saved best.pt')

    scheduler.step()

    record = {'epoch': epoch, 'train_loss': train_loss, 'lr': optimizer.param_groups[0]['lr'], **metrics}
    history.append(record)
    trainer._log(record, score)

print(f'\nTraining complete. Best macro AUC: {trainer.best_metric:.4f}')

## 8 · Evaluation on test set

In [ ]:
# Load best weights
import torch
best_ckpt = torch.load(f'{CKPT_DIR}/best.pt', map_location=trainer.device)
expert.load_state_dict(best_ckpt['model'])
print(f'Loaded best checkpoint from epoch {best_ckpt["epoch"]} (best AUC={best_ckpt["best_metric"]:.4f})')

test_samples = load_chest_xray14(DATASET_ROOT, split='test', val_fraction=VAL_FRACTION)
test_ds      = ScanDataset(test_samples, val_pre, modality=Modality.XRAY, body_part=BodyPart.CHEST)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f'Test samples: {len(test_samples):,}')

test_eval = MultilabelEvaluator(class_names=list(CHESTXRAY14_LABELS))
expert.eval()
from training.losses import _classification_logits
device = trainer.device
with torch.no_grad():
    for x, targets in test_loader:
        x = x.to(device)
        targets = {k: v.to(device) for k, v in targets.items()}
        outputs = expert(x)
        logits  = _classification_logits(outputs)
        if logits is not None and 'label' in targets:
            test_eval.update(logits, targets['label'])

test_metrics = test_eval.compute()
print(f"\nTest macro AUC : {test_metrics['auc']:.4f}")
print(f"Macro sensitivity: {test_metrics['macro_sensitivity']:.4f}")
print(f"Macro specificity: {test_metrics['macro_specificity']:.4f}")
print(f"ECE              : {test_metrics['ece']:.4f}")

print('\nPer-label AUC:')
for label, auc in sorted(test_metrics['auc_per_label'].items(), key=lambda x: -(x[1] or 0)):
    auc_str = f'{auc:.4f}' if auc is not None else 'n/a'
    sens = test_metrics['sensitivity_per_label'].get(label, float('nan'))
    print(f'  {label:<22} AUC={auc_str}  sens={sens:.3f}')

## 9 · Plot training curves

In [ ]:
import matplotlib.pyplot as plt

epochs_x  = [r['epoch'] for r in history]
train_loss = [r['train_loss'] for r in history]
val_loss   = [r.get('val_loss', float('nan')) for r in history]
auc_curve  = [r.get('auc') or float('nan') for r in history]
lr_curve   = [r.get('lr', 0) for r in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_x, train_loss, label='train')
axes[0].plot(epochs_x, val_loss,   label='val')
axes[0].set(title='BCE Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()

axes[1].plot(epochs_x, auc_curve, color='green')
axes[1].set(title='Val Macro AUC', xlabel='Epoch', ylabel='AUC')
axes[1].axhline(test_metrics['auc'], color='orange', linestyle='--', label=f'test={test_metrics["auc"]:.3f}')
axes[1].legend()

axes[2].plot(epochs_x, lr_curve, color='purple')
axes[2].set(title='Learning Rate (cosine)', xlabel='Epoch', ylabel='LR')
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/training_curves.png', dpi=120)
plt.show()

## 10 · Sample report generation

In [ ]:
import random
from ingest.loaders import load_scan
from reporting import findings_from_classification, Reporter, GridZoneLocalizer
from explainability import GradCAM

# Pick a random test image that has at least one finding
positive_samples = [s for s in test_samples if sum(s.label) > 0]
sample = random.choice(positive_samples[:200])  # first 200 for speed
print('Image:', os.path.basename(sample.path))
true_labels = [CHESTXRAY14_LABELS[i] for i, v in enumerate(sample.label) if v > 0]
print('True labels:', true_labels)

# Full predict pipeline
scan = load_scan(sample.path, modality=Modality.XRAY, body_part=BodyPart.CHEST)
pred = expert.predict(scan)
print('\nModel probs (top 5):')
for k, v in sorted(pred.class_probs.items(), key=lambda x: -x[1])[:5]:
    print(f'  {k:<22} {v:.3f}')

# Grad-CAM heatmaps for localization
cam = GradCAM(expert)
heatmaps = cam.for_labels(scan.data)

# Findings from multi-label scores + Grad-CAM zones
findings = findings_from_classification(
    pred.class_probs,
    thresholds=0.5,
    confidence=pred.confidence,
    heatmaps=heatmaps,
    localizer=GridZoneLocalizer(),
)

print(f'\nFindings ({len(findings)}):')
for f in findings:
    print(' ', f.to_facts())

# Report — uses ANTHROPIC_API_KEY env var if set, otherwise deterministic template
import os
if 'ANTHROPIC_API_KEY' not in os.environ:
    print('\nNote: ANTHROPIC_API_KEY not set — using deterministic template reporter.')
    print('Set it in Colab Secrets (key icon) to enable LLM-polished prose.')

reporter = Reporter(auto_llm=True)
report   = reporter.report(findings, scan.meta)

print(f'\n=== REPORT (generator: {report.generator}) ===')
print(report.to_text())

## 11 · Visualize Grad-CAM heatmap

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Show original image + heatmaps for the top 3 predicted findings
top_findings = [f for f in findings if f.present][:3]
n = len(top_findings) + 1
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

# Decode image for display
img_np = scan.data.permute(1, 2, 0).cpu().numpy()
img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
if img_np.shape[2] == 1:
    img_np = img_np[:, :, 0]
    cmap = 'gray'
else:
    cmap = None

axes[0].imshow(img_np, cmap=cmap)
axes[0].set_title('Input X-ray')
axes[0].axis('off')

for ax, finding in zip(axes[1:], top_findings):
    heat = heatmaps.get(finding.label)
    if heat is not None:
        ax.imshow(img_np, cmap=cmap)
        ax.imshow(heat.numpy(), alpha=0.45, cmap='jet', vmin=0, vmax=1)
        ax.set_title(f'{finding.label}\np={finding.probability:.2f}')
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{CKPT_DIR}/gradcam_sample.png', dpi=120)
plt.show()